In [1]:
import pandas as pd
import os
import numpy as np
from datetime import datetime, timedelta
import re

In [2]:
pd.set_option('display.max_columns', None)

#### <span style="color:#FF6347;">**READ**</span> 

In [3]:
# file_path = "P:/Dataset/MO-DBT-data-curation/Cancer/mo_cancer.xlsx"
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "mo_cancer_candidate" + ".xlsx")
mo_cancer = pd.read_excel(file_path)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "normal_candidate" + ".xlsx")
normal_candidate = pd.read_excel(file_path)

file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "mo_cancer_candidate_ehr_@_index_1" + ".xlsx")
mo_cancer_ehr = pd.read_excel(file_path)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "normal_candidate_ehr_@_index_1" + ".xlsx")
normal_candidate_ehr = pd.read_excel(file_path)

In [4]:
mo_cancer[(mo_cancer["Index"] == "INDEX-1") & (mo_cancer["Series"] == "DBT")]["PATIENT_STUDY_ID"].nunique(), mo_cancer[(mo_cancer["Index"] == "INDEX") & (mo_cancer["Series"] == "DBT")]["PATIENT_STUDY_ID"].nunique()

(653, 648)

In [5]:
normal_candidate[(normal_candidate["Index"] == "INDEX-1") & (normal_candidate["Series"] == "DBT")]["PATIENT_STUDY_ID"].nunique(), normal_candidate[(normal_candidate["Index"] == "INDEX") & (normal_candidate["Series"] == "DBT")]["PATIENT_STUDY_ID"].nunique()

(30903, 30789)

# **Step 1. BI-RADS**

In [6]:
mo_cancer["COMPOSITION_NAME"].unique(), normal_candidate["COMPOSITION_NAME"].unique()

(array(['Heterogeneously dense (51% - 75%)', 'Extremely dense (>75%)',
        'Scattered fibroglandular (25% - 50%)', nan], dtype=object),
 array(['Heterogeneously dense (51% - 75%)',
        'Scattered fibroglandular (25% - 50%)', 'Extremely dense (>75%)',
        'Almost entirely fat (<25%)'], dtype=object))

In [7]:
# ── STEP 1: Bilateral BIRADS combo + Density at INDEX-1 ───────────────────────
BIRADS_SHORT = {'1 - Negative': '1', '2 - Benign finding': '2'}

DENSITY_SHORT = {
    'Almost entirely fat (<25%)': 'A',
    'Scattered fibroglandular (25% - 50%)': 'B',
    'Heterogeneously dense (51% - 75%)': 'C',
    'Extremely dense (>75%)': 'D'
}

c1 = mo_cancer[mo_cancer['Index'] == 'INDEX-1'].copy()
n1 = normal_candidate[normal_candidate['Index'] == 'INDEX-1'].copy()

def get_bilateral_combo(df):
    df = df.copy()
    df['CAT'] = df['FINDING_CATEGORY'].map(BIRADS_SHORT)

    side_dedup = df.drop_duplicates(subset=['PATIENT_STUDY_ID', 'SIDE'], keep='first')
    valid = side_dedup[side_dedup['SIDE'].isin(['L', 'R'])]
    piv = valid.pivot_table(
        index='PATIENT_STUDY_ID', columns='SIDE', values='CAT', aggfunc='first'
    )
    piv.columns.name = None
    def combo(row):
        vals = sorted([v for v in row.dropna()])
        return '+'.join(vals) if vals else None
    piv['birads_combo'] = piv.apply(combo, axis=1)
    result = piv[['birads_combo']].reset_index()

    nan_side_pids = set(df['PATIENT_STUDY_ID']) - set(result['PATIENT_STUDY_ID'])
    if nan_side_pids:
        nan_side = (
            df[df['PATIENT_STUDY_ID'].isin(nan_side_pids)]
            .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
            [['PATIENT_STUDY_ID', 'CAT']]
            .rename(columns={'CAT': 'birads_combo'})
        )
        result = pd.concat([result, nan_side], ignore_index=True)

    # ── Add density (one value per patient; take first non-null) ──────────────
    density = (
        df.dropna(subset=['COMPOSITION_NAME'])          # ← adjust col name if needed
        .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
        [['PATIENT_STUDY_ID', 'COMPOSITION_NAME']]
    )
    density['density'] = density['COMPOSITION_NAME'].map(DENSITY_SHORT)
    result = result.merge(density[['PATIENT_STUDY_ID', 'density']], on='PATIENT_STUDY_ID', how='left')

    return result

def combo_table(df, label, var):
    n_total = df['PATIENT_STUDY_ID'].nunique()
    counts  = df[var].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'Combo': counts.index,
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*45}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*45}")
    print(tbl.to_string(index=False))
    print()

In [8]:
c_combo = get_bilateral_combo(c1)
n_combo = get_bilateral_combo(n1)

In [9]:
combo_table(c_combo, 'MO Cancer', 'birads_combo')
combo_table(n_combo, 'Normal Candidate', 'birads_combo')

─────────────────────────────────────────────
MO Cancer  (n=653)
─────────────────────────────────────────────
Combo  Count    %
  2+2    490 75.0
  1+1    144 22.1
    2     13  2.0
    1      6  0.9

─────────────────────────────────────────────
Normal Candidate  (n=30,903)
─────────────────────────────────────────────
Combo  Count    %
  2+2  17912 58.0
  1+1   9461 30.6
    1   3288 10.6
    2    242  0.8



In [10]:
combo_table(c_combo, 'MO Cancer', 'density')
combo_table(n_combo, 'Normal Candidate', 'density')

─────────────────────────────────────────────
MO Cancer  (n=653)
─────────────────────────────────────────────
Combo  Count    %
    C    597 91.4
    B     48  7.4
    D      8  1.2

─────────────────────────────────────────────
Normal Candidate  (n=30,903)
─────────────────────────────────────────────
Combo  Count    %
    C  27675 89.6
    B   2359  7.6
    D    862  2.8
    A      7  0.0



# **Step 2. Race**

In [11]:
c_race_cols = [c for c in mo_cancer_ehr.columns if c.startswith('RACE_')]
n_race_cols = [c for c in normal_candidate_ehr.columns if c.startswith('RACE_')]

In [12]:
ASIAN_GROUPS = {
    'KOREAN', 'OTHER ASIAN', 'VIETNAMESE', 'CHINESE', 'FILIPINO',
    'JAPANESE', 'INDIAN (ASIAN)'
}

PACIFIC_GROUPS = {
    'HAWAIIAN',  'OTHER PACIFIC ISLANDER'
}

def get_simplified_race(df):
    race_cols = [c for c in df.columns if c.startswith('RACE_')]
    df2 = df[race_cols].fillna(0)
    all_zero = df2.sum(axis=1) == 0
    race = df2.idxmax(axis=1).str.replace('RACE_', '')
    race[all_zero] = 'UNKNOWN'
    def simplify(r):
        if r == 'WHITE':  return 'WHITE'
        if r == 'BLACK':  return 'BLACK'
        if r == 'AMERICAN INDIAN/ALASKA NATIVE':  return 'AMERICAN INDIAN/ALASKA NATIVE'
        if r in PACIFIC_GROUPS: return 'HAWAIIAN/PACIFIC ISLANDER'
        if r in ASIAN_GROUPS: return 'ASIAN'
        return 'OTHER/UNKNOWN'
    return race.apply(simplify)

mo_cancer_ehr['race']        = get_simplified_race(mo_cancer_ehr)
normal_candidate_ehr['race'] = get_simplified_race(normal_candidate_ehr)

In [13]:
def race_summary(df, label):
    n_total = len(df)
    counts = df['race'].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'Race':  counts.index,
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*40}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*40}")
    print(tbl.to_string(index=False))
    print()

race_summary(mo_cancer_ehr,        'MO Cancer')
race_summary(normal_candidate_ehr, 'Normal Candidate')

────────────────────────────────────────
MO Cancer  (n=664)
────────────────────────────────────────
                         Race  Count    %
                        WHITE    610 91.9
                        BLACK     31  4.7
                        ASIAN     15  2.3
                OTHER/UNKNOWN      7  1.1
AMERICAN INDIAN/ALASKA NATIVE      1  0.2

────────────────────────────────────────
Normal Candidate  (n=30,901)
────────────────────────────────────────
                         Race  Count    %
                        WHITE  27833 90.1
                        BLACK   1496  4.8
                OTHER/UNKNOWN    784  2.5
                        ASIAN    753  2.4
AMERICAN INDIAN/ALASKA NATIVE     23  0.1
    HAWAIIAN/PACIFIC ISLANDER     12  0.0



In [14]:
mo_cancer_ehr['PATIENT_STUDY_ID'].nunique(), normal_candidate_ehr['PATIENT_STUDY_ID'].nunique()

(653, 30901)

# **Step 3. Age**

In [15]:
mo_cancer_ehr['PatientAge'].min(), mo_cancer_ehr['PatientAge'].max()

(38.0, 88.0)

In [16]:
AGE_BINS   = list(range(40, 90, 5))
AGE_LABELS = [f'{b}-{b+4}' for b in AGE_BINS[:-1]]

mo_cancer_ehr['age_bin']        = pd.cut(mo_cancer_ehr['PatientAge'],        bins=AGE_BINS, labels=AGE_LABELS, right=False)
normal_candidate_ehr['age_bin'] = pd.cut(normal_candidate_ehr['PatientAge'], bins=AGE_BINS, labels=AGE_LABELS, right=False)

In [17]:
def age_summary(df, label):
    n_total = len(df)
    counts = df['age_bin'].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'AGE_LABELS':  counts.index,
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*40}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*40}")
    print(tbl.to_string(index=False))
    print()

age_summary(mo_cancer_ehr,        'MO Cancer')
age_summary(normal_candidate_ehr, 'Normal Candidate')

────────────────────────────────────────
MO Cancer  (n=664)
────────────────────────────────────────
AGE_LABELS  Count    %
     60-64    127 19.1
     65-69    100 15.1
     55-59     92 13.9
     70-74     92 13.9
     50-54     66  9.9
     45-49     58  8.7
     75-79     50  7.5
     40-44     33  5.0
     80-84     30  4.5
       NaN     16  2.4

────────────────────────────────────────
Normal Candidate  (n=30,901)
────────────────────────────────────────
AGE_LABELS  Count    %
     60-64   4826 15.6
     65-69   4359 14.1
     55-59   4354 14.1
     50-54   3967 12.8
     70-74   3593 11.6
     45-49   3508 11.4
     40-44   2623  8.5
     75-79   1908  6.2
     80-84    950  3.1
       NaN    813  2.6



# **Step 4. Match**

In [18]:
# ── Config ────────────────────────────────────────────────────────────────────
RANDOM_SEED = 42
N_MATCH     = 8  # normals per cancer patient

In [19]:
# ── Feature tables ────────────────────────────────────────────────────
c_feat = c_combo.merge(mo_cancer_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')
n_feat = n_combo.merge(normal_candidate_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')

def make_stratum(df):
    return df['birads_combo'].astype(str) + '|' + df['density'].astype(str) + '|' + df['race'].astype(str) + '|' + df['age_bin'].astype(str)

c_feat['stratum'] = make_stratum(c_feat)
n_feat['stratum'] = make_stratum(n_feat)

# ── Filter out unresolvable cancer entries before matching ────────────────────
c_feat_match = c_feat[
    c_feat['age_bin'].notna() &
    (c_feat['race'] != 'OTHER/UNKNOWN') &
    (c_feat['birads_combo'].isin(['1+1', '2+2'])) &
    (c_feat['density'].isin(['C', 'D']))
].copy()

print(f"Cancer patients used for matching : {len(c_feat_match)} / {len(c_feat)}")
print(f"Excluded (NaN age or unknown race): {len(c_feat) - len(c_feat_match)}")

Cancer patients used for matching : 579 / 664
Excluded (NaN age or unknown race): 85


In [20]:
c_feat_match

,PATIENT_STUDY_ID,birads_combo,density,PatientAge,age_bin,race,stratum
0,4330344296,2+2,C,65.0,65-69,WHITE,2+2|C|WHITE|65-69
1,4330358672,2+2,C,49.0,45-49,WHITE,2+2|C|WHITE|45-49
2,4330421371,1+1,C,45.0,45-49,ASIAN,1+1|C|ASIAN|45-49
3,4330449572,2+2,C,70.0,70-74,WHITE,2+2|C|WHITE|70-74
4,4330495728,2+2,C,56.0,55-59,WHITE,2+2|C|WHITE|55-59
...,...,...,...,...,...,...,...
637,4337127693,1+1,C,66.0,65-69,WHITE,1+1|C|WHITE|65-69
638,4337129380,2+2,C,69.0,65-69,ASIAN,2+2|C|ASIAN|65-69
639,4337511911,2+2,C,71.0,70-74,WHITE,2+2|C|WHITE|70-74
641,4337779562,2+2,C,71.0,70-74,WHITE,2+2|C|WHITE|70-74


In [21]:
c_feat_match['birads_combo'].unique(), c_feat_match['density'].unique(), c_feat_match['age_bin'].unique(), c_feat_match['race'].unique()

(array(['2+2', '1+1'], dtype=object),
 array(['C', 'D'], dtype=object),
 ['65-69', '45-49', '70-74', '55-59', '40-44', '60-64', '50-54', '75-79', '80-84']
 Categories (9, object): ['40-44' < '45-49' < '50-54' < '55-59' ... '65-69' < '70-74' < '75-79' < '80-84'],
 array(['WHITE', 'ASIAN', 'BLACK', 'AMERICAN INDIAN/ALASKA NATIVE'],
       dtype=object))

In [22]:
# ── Match ─────────────────────────────────────────────────────────────
rng = np.random.default_rng(RANDOM_SEED)
normal_pool = n_feat.copy()
normal_pool_by_stratum = {s: grp.index.tolist() for s, grp in normal_pool.groupby('stratum')}

matched_cancer_rows, matched_normal_rows, unmatched_cancer_ids = [], [], []
used_normal_indices = set()

for _, cancer_row in c_feat_match.iterrows():
    stratum   = cancer_row['stratum']
    available = [i for i in normal_pool_by_stratum.get(stratum, []) if i not in used_normal_indices]

    # Fallback: relax age bin ±5 then ±10 years if stratum is empty
    if len(available) == 0:
        birads, density, race, age_bin = stratum.split('|')
        age_idx = AGE_LABELS.index(age_bin) if age_bin in AGE_LABELS else -1
        for offset in [-1, 1, -2, 2]:
            adj_idx = age_idx + offset
            if 0 <= adj_idx < len(AGE_LABELS):
                adj_stratum = f'{birads}|{density}|{race}|{AGE_LABELS[adj_idx]}'
                available += [i for i in normal_pool_by_stratum.get(adj_stratum, []) if i not in used_normal_indices]

    if len(available) == 0:
        unmatched_cancer_ids.append(cancer_row['PATIENT_STUDY_ID'])
        continue

    chosen_idx = rng.choice(available, size=min(N_MATCH, len(available)), replace=False).tolist()
    used_normal_indices.update(chosen_idx)
    matched_cancer_rows.append(cancer_row)
    for idx in chosen_idx:
        row = normal_pool.loc[idx].copy()
        row['matched_cancer_id'] = cancer_row['PATIENT_STUDY_ID']
        matched_normal_rows.append(row)

matched_cancer_list  = pd.DataFrame(matched_cancer_rows).reset_index(drop=True)
matched_normal_list = pd.DataFrame(matched_normal_rows).reset_index(drop=True)

# ── Summary ───────────────────────────────────────────────────────────
print(f"Cancer matched    : {len(matched_cancer_list)} / {len(c_feat_match)}")
print(f"Normals selected  : {len(matched_normal_list)}  (ratio {len(matched_normal_list)/len(matched_cancer_list):.2f}:1)")
print(f"Unmatched IDs     : {unmatched_cancer_ids}")

Cancer matched    : 579 / 579
Normals selected  : 4622  (ratio 7.98:1)
Unmatched IDs     : []


In [23]:
# ── Side-by-side distribution summary: Cancer vs Matched Normals ──────────────

def dist_summary(df, label):
    n = len(df)
    print(f"\n{'='*50}")
    print(f"{label}  (n={n})")
    print(f"{'='*50}")

    print("\n── BIRADS Combo ──────────────────────────────")
    for k, v in df['birads_combo'].value_counts(dropna=False).items():
        print(f"  {str(k):15s}: {v:4d}  ({v/n*100:.1f}%)")

    print("\n── Density ───────────────────────────────────")
    for k, v in df['density'].value_counts(dropna=False).items():
        print(f"  {str(k):15s}: {v:4d}  ({v/n*100:.1f}%)")

    print("\n── Age Bin ───────────────────────────────────")
    for k, v in df['age_bin'].value_counts(dropna=False).sort_index().items():
        print(f"  {str(k):15s}: {v:4d}  ({v/n*100:.1f}%)")

    print("\n── Race ──────────────────────────────────────")
    for k, v in df['race'].value_counts(dropna=False).items():
        print(f"  {str(k):30s}: {v:4d}  ({v/n*100:.1f}%)")

dist_summary(c_feat_match,       'MO Cancer')
dist_summary(matched_normal_list, 'Matched Normals')


MO Cancer  (n=579)

── BIRADS Combo ──────────────────────────────
  2+2            :  447  (77.2%)
  1+1            :  132  (22.8%)

── Density ───────────────────────────────────
  C              :  570  (98.4%)
  D              :    9  (1.6%)

── Age Bin ───────────────────────────────────
  40-44          :   32  (5.5%)
  45-49          :   56  (9.7%)
  50-54          :   58  (10.0%)
  55-59          :   81  (14.0%)
  60-64          :  115  (19.9%)
  65-69          :   87  (15.0%)
  70-74          :   81  (14.0%)
  75-79          :   44  (7.6%)
  80-84          :   25  (4.3%)

── Race ──────────────────────────────────────
  WHITE                         :  536  (92.6%)
  BLACK                         :   27  (4.7%)
  ASIAN                         :   15  (2.6%)
  AMERICAN INDIAN/ALASKA NATIVE :    1  (0.2%)

Matched Normals  (n=4622)

── BIRADS Combo ──────────────────────────────
  2+2            : 3568  (77.2%)
  1+1            : 1054  (22.8%)

── Density ──────────────────────

In [24]:
matched_normal_list

,PATIENT_STUDY_ID,birads_combo,density,PatientAge,age_bin,race,stratum,matched_cancer_id
0,4333093218,2+2,C,67.0,65-69,WHITE,2+2|C|WHITE|65-69,4330344296
1,4334283603,2+2,C,67.0,65-69,WHITE,2+2|C|WHITE|65-69,4330344296
2,4334557263,2+2,C,69.0,65-69,WHITE,2+2|C|WHITE|65-69,4330344296
3,4334090521,2+2,C,68.0,65-69,WHITE,2+2|C|WHITE|65-69,4330344296
4,4333724506,2+2,C,65.0,65-69,WHITE,2+2|C|WHITE|65-69,4330344296
...,...,...,...,...,...,...,...,...
4617,4334831489,2+2,C,60.0,60-64,WHITE,2+2|C|WHITE|60-64,4339604670
4618,4334473162,2+2,C,63.0,60-64,WHITE,2+2|C|WHITE|60-64,4339604670
4619,4333002791,2+2,C,61.0,60-64,WHITE,2+2|C|WHITE|60-64,4339604670
4620,4333245642,2+2,C,61.0,60-64,WHITE,2+2|C|WHITE|60-64,4339604670


In [25]:
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "matched_correspondance" + ".xlsx")
matched_normal_list.to_excel(file_path, index=False)

In [26]:
matched_normal_candidate_ehr = normal_candidate_ehr[
    normal_candidate_ehr['PATIENT_STUDY_ID'].isin(matched_normal_list['PATIENT_STUDY_ID'])
].copy()

print(f"Matched normal EHR rows : {len(matched_normal_candidate_ehr)}")
print(f"Unique patients         : {matched_normal_candidate_ehr['PATIENT_STUDY_ID'].nunique()}")

Matched normal EHR rows : 4622
Unique patients         : 4622


In [27]:
matched_normal_candidate = normal_candidate[
    normal_candidate['PATIENT_STUDY_ID'].isin(matched_normal_list['PATIENT_STUDY_ID'])
].copy()

print(f"Matched normal EHR rows : {len(matched_normal_candidate)}")
print(f"Unique patients         : {matched_normal_candidate['PATIENT_STUDY_ID'].nunique()}")

Matched normal EHR rows : 18326
Unique patients         : 4622


In [28]:
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "matched_normal_candidate_ehr_@_index_1" + ".xlsx")
matched_normal_candidate_ehr.to_excel(file_path, index=False)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "matched_normal_candidate" + ".xlsx")
matched_normal_candidate.to_excel(file_path, index=False)